In [ ]:
import pickle
import numpy as np
from dataset_form import SimpleDataset, train_transform, val_transform

# Cargar datos
with open('data_minimal.pkl', 'rb') as f:
    data = pickle.load(f)

print("Filtrando imágenes con anotaciones...")
print("="*60)

# Función para verificar si una imagen tiene anotaciones
def tiene_anotaciones(tile_id, annotations_dict):
    if tile_id not in annotations_dict:
        return False
    
    annotations = annotations_dict[tile_id].get('annotations', [])
    
    # Verificar si tiene blood_vessel (clase 1)
    for ann in annotations:
        if ann.get('type') == 'blood_vessel':
            return True
    return False

# Filtrar train y val
train_con_anotaciones = [
    tid for tid in data['train'] 
    if tiene_anotaciones(tid, data['annotations'])
]

val_con_anotaciones = [
    tid for tid in data['val'] 
    if tiene_anotaciones(tid, data['annotations'])
]

print(f"\n📊 ESTADÍSTICAS:")
print(f"   Train original: {len(data['train'])}")
print(f"   Train con anotaciones: {len(train_con_anotaciones)} ({100*len(train_con_anotaciones)/len(data['train']):.1f}%)")
print(f"\n   Val original: {len(data['val'])}")
print(f"   Val con anotaciones: {len(val_con_anotaciones)} ({100*len(val_con_anotaciones)/len(data['val']):.1f}%)")

# Guardar nuevo pickle
data_filtrado = {
    'train': train_con_anotaciones,
    'val': val_con_anotaciones,
    'annotations': data['annotations']
}

with open('data_minimal_filtrado.pkl', 'wb') as f:
    pickle.dump(data_filtrado, f)

print(f"\n✓ Guardado: data_minimal_filtrado.pkl")

# Verificar algunos ejemplos
print(f"\n🔍 VERIFICACIÓN (primeras 5 del val filtrado):")
print("="*60)

val_dataset = SimpleDataset(
    val_con_anotaciones[:5],
    '../train/',
    data['annotations'],
    val_transform
)

for i in range(5):
    img, mask, tile_id = val_dataset[i]
    unique_vals = np.unique(mask.numpy())
    tiene_ann = 1 in unique_vals
    
    print(f"{i+1}. {tile_id}")
    print(f"   Valores únicos: {unique_vals}")
    print(f"   {'✓ Tiene blood vessels' if tiene_ann else '⚠️  Sin blood vessels'}")

## En sus códigos solo modifican esto 

In [ ]:
# ❌ ANTES:
with open('data_minimal.pkl', 'rb') as f:
    data = pickle.load(f)

# ✅ DESPUÉS:
with open('data_minimal_filtrado.pkl', 'rb') as f:
    data = pickle.load(f)
